# 00 — Setup, download, raw AnnData, and SpatialData (interactive)

This notebook replaces `%run` with inspectable cells. Run one section at a time. Expensive write operations are controlled by explicit `WRITE_*` switches.

In [24]:
# ============================================================
# Dataset_02 CosMx — Jupyter initialization
# ============================================================

from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc


# ------------------------------------------------------------
# Robustly locate project root
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """
    Search current directory and its parents for the project root.

    A valid project root contains:
        src/
        data/
    """

    for candidate in [start, *start.parents]:

        if (
            (candidate / "src").is_dir()
            and
            (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root.\n"
        f"Starting directory: {start}\n\n"
        "Expected a parent directory containing:\n"
        "  src/\n"
        "  data/"
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)


# ------------------------------------------------------------
# Make src importable
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ------------------------------------------------------------
# Common paths
# ------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

SPATIAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "spatial"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

CONFIG_DIR = (
    PROJECT_ROOT
    / "config"
)


# ------------------------------------------------------------
# Scanpy settings
# ------------------------------------------------------------

sc.settings.verbosity = 2


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("Dataset_02 CosMx Jupyter environment")
print("=" * 70)

print("Notebook directory :", CURRENT_DIR)
print("Project root       :", PROJECT_ROOT)
print("Python             :", sys.executable)

print()

print(
    "src exists         :",
    (PROJECT_ROOT / "src").exists(),
)

print(
    "raw data exists    :",
    RAW_DIR.exists(),
)

print(
    "processed exists   :",
    PROCESSED_DIR.exists(),
)

print(
    "spatial exists     :",
    SPATIAL_DIR.exists(),
)

Dataset_02 CosMx Jupyter environment
Notebook directory : /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/jupyter-2
Project root       : /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised
Python             : /home/jqu/.conda/envs/spatialdata/bin/python

src exists         : True
raw data exists    : True
processed exists   : True
spatial exists     : True


In [25]:
import spatialdata, dask, pandas, zarr, anndata
import spatialdata_plot
print("spatialdata     :", spatialdata.__version__)
print("spatialdata-plot:", spatialdata_plot.__version__)
print("scanpy          :", sc.__version__)
print("anndata         :", anndata.__version__)
print("dask            :", dask.__version__)
print("pandas          :", pandas.__version__)
print("zarr            :", zarr.__version__)

spatialdata     : 0.7.2
spatialdata-plot: 0.3.3
scanpy          : 1.11.5
anndata         : 0.12.11
dask            : 2026.1.1
pandas          : 2.3.3
zarr            : 3.1.6


/lsf_tmp/323066199.tmpdir/ipykernel_1573392/2800429130.py:5: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy          :", sc.__version__)
/lsf_tmp/323066199.tmpdir/ipykernel_1573392/2800429130.py:6: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata         :", anndata.__version__)


## A. Programmatic GEO download
Set `DOWNLOAD=True` only when you want to download/check the five GEO files.

In [26]:
from urllib.request import urlretrieve
from src.cosmx_io import FILES, BASE_URL, validate_gzip_csv

RAW_DIR = PROJECT_ROOT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD = False

if DOWNLOAD:
    for label, filename in FILES.items():
        path = RAW_DIR / filename
        if path.exists():
            try:
                validate_gzip_csv(path)
                print("OK/skip:", label, path.name, f"({path.stat().st_size/1e6:.1f} MB)")
                continue
            except Exception as e:
                print("Existing file invalid; redownloading:", path, e)
        url = BASE_URL + filename
        print("Downloading", url)
        urlretrieve(url, path)
        validate_gzip_csv(path)
        print("Saved:", path)
else:
    print("DOWNLOAD=False; existing raw files:")
    for label, filename in FILES.items():
        p = RAW_DIR / filename
        print(f"  {label:12s}", p.exists(), p)

DOWNLOAD=False; existing raw files:
  expression   True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_exprMat_file.csv.gz
  metadata     True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_metadata_file.csv.gz
  fov          True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_fov_positions_file.csv.gz
  polygons     True /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/raw/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_polygons.csv.gz
  transcripts  True /research/rgs01/home/clus

## B. Build and inspect raw AnnData
This calls the reusable `src.cosmx_io.build_raw_anndata()` function directly, not a production script.

In [27]:
from src.cosmx_io import build_raw_anndata

ADATA_PATH = PROJECT_ROOT / "data/processed/GSM9046088_CosMx_raw.h5ad"
WRITE_ANNDATA = False

adata_raw = build_raw_anndata(RAW_DIR)
print(adata_raw)
display(adata_raw.obs.head())
print("spatial:", adata_raw.obsm.get("spatial", np.empty((0,2))).shape)
print("counts layer:", "counts" in adata_raw.layers)

#if WRITE_ANNDATA:
#    ADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
#    adata_raw.write_h5ad(ADATA_PATH, compression="gzip")
#    print("Saved:", ADATA_PATH)

AnnData object with n_obs × n_vars = 43565 × 1010
    obs: 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.B2M.MembraneStain', 'Max.B2M.MembraneStain', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI'
    uns: 'spatial_coordinate_columns', 'fov_positions', 'GEO_accession', 'platform', 'sample'
    obsm: 'spatial'
    layers: 'counts'


,fov,cell_ID,Area,AspectRatio,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px,Width,Height,Mean.PanCK,Max.PanCK,Mean.CD68,Max.CD68,Mean.B2M.MembraneStain,Max.B2M.MembraneStain,Mean.CD45,Max.CD45,Mean.DAPI,Max.DAPI
unique_cell_id,,,,,,,,,,,,,,,,,,,,
fov_1_cell_1,1,1,7538,0.95,2515,4056,-495476.6667,11339.33333,99,104,1183,4581,0,105,108,458,2,128,1,87
fov_1_cell_2,1,2,4969,1.11,3831,3925,-494160.6667,11208.33333,83,75,143,1746,0,93,53,787,3,530,34,306
fov_1_cell_3,1,3,4045,0.75,2368,3901,-495623.6667,11184.33333,65,87,31,378,5,829,344,1389,18,1284,105,397
fov_1_cell_4,1,4,7853,0.86,2921,3695,-495070.6667,10978.33333,96,111,1041,8586,18,245,172,843,29,589,136,550
fov_1_cell_5,1,5,7992,0.81,3907,3665,-494084.6667,10948.33333,90,111,889,7437,6,310,96,1015,16,909,0,34


spatial: (43565, 2)
counts layer: True


## C. Build SpatialData interactively
This is the same logic as `02_build_sdata.py`, exposed as cells. It can be memory-intensive because the transcript CSV is large.

In [28]:
import dask.dataframe as dd
import geopandas as gpd
from shapely.geometry import Polygon
from spatialdata import SpatialData
from spatialdata.models import PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity
from src.cosmx_io import unique_cell_id

POLY = RAW_DIR / FILES["polygons"]
TX = RAW_DIR / FILES["transcripts"]

poly = pd.read_csv(POLY)
cell_col = "cell_ID" if "cell_ID" in poly.columns else "cellID"
poly["unique_cell_id"] = unique_cell_id(
    pd.to_numeric(poly["fov"]).astype(int),
    pd.to_numeric(poly[cell_col]).astype(int),
)
print(poly.shape)
display(poly.head())

(1197279, 8)


,Unnamed: 0,fov,cellID,x_local_px,y_local_px,x_global_px,y_global_px,unique_cell_id
0,1,1,1,2515,4107,-495476.666667,11390.333333,fov_1_cell_1
1,2,1,1,2522,4106,-495469.666667,11389.333333,fov_1_cell_1
2,3,1,1,2527,4105,-495464.666667,11388.333333,fov_1_cell_1
3,4,1,1,2536,4102,-495455.666667,11385.333333,fov_1_cell_1
4,5,1,1,2538,4101,-495453.666667,11384.333333,fov_1_cell_1


In [29]:
# Build polygons. This may take a little time.
records = []
invalid_fixed = skipped = 0
for uid, g in poly.groupby("unique_cell_id", sort=False):
    xy = g[["x_global_px", "y_global_px"]].to_numpy(float)
    if len(xy) < 3:
        skipped += 1; continue
    geom = Polygon(xy)
    if not geom.is_valid:
        geom = geom.buffer(0); invalid_fixed += 1
    if geom.is_empty:
        skipped += 1; continue
    records.append((uid, geom))

shapes = gpd.GeoDataFrame(
    {"geometry": [x[1] for x in records]},
    index=pd.Index([x[0] for x in records], name="instance_id"), crs=None,
)
shapes_model = ShapesModel.parse(shapes, transformations={"global": Identity()})
print("valid polygons:", len(shapes), "fixed:", invalid_fixed, "skipped:", skipped)

valid polygons: 43565 fixed: 0 skipped: 0


In [30]:
# Build SpatialData table from raw AnnData and matching polygons.
tab_adata = adata_raw.copy()
tab_adata.obs["region"] = pd.Categorical(["cell_boundaries"] * tab_adata.n_obs)
tab_adata.obs["instance_id"] = tab_adata.obs_names.astype(str)
keep = tab_adata.obs_names.intersection(shapes.index)
tab_adata = tab_adata[keep].copy()
table = TableModel.parse(
    tab_adata, region="cell_boundaries", region_key="region", instance_key="instance_id"
)
print("table:", table.shape)

table: (43565, 1010)


In [31]:
# Transcript points. Explicit dtypes avoid the CellComp inference bug.
preview = pd.read_csv(TX, nrows=100)
dtype_map = {"target": "object"}
if "CellComp" in preview.columns:
    dtype_map["CellComp"] = "object"

tx = dd.read_csv(TX, blocksize=None, assume_missing=True, dtype=dtype_map)
points = PointsModel.parse(
    tx,
    coordinates={"x": "x_global_px", "y": "y_global_px"},
    feature_key="target",
    transformations={"global": Identity()},
)
print("Transcript columns:", list(tx.columns))
print(tx.dtypes)

INFO     Column `z` in `data` will be ignored since the data is 2D.                                                
WARNING  The `feature_key` column target is categorical with unknown categories. Please ensure the categories are  
         known before calling `PointsModel.parse()` to avoid significant performance implications due to the need  
         for dask to compute the categories. If you did not use PointsModel.parse() explicitly in your code (e.g.  
         this message is coming from a reader in `spatialdata_io`), please report this finding.                    
Transcript columns: ['Unnamed: 0', 'fov', 'cell_ID', 'x_global_px', 'y_global_px', 'x_local_px', 'y_local_px', 'z', 'target', 'CellComp']
Unnamed: 0             float64
fov                    float64
cell_ID                float64
x_global_px            float64
y_global_px            float64
x_local_px             float64
y_local_px             float64
z                      float64
target         string[pyarrow]
Cel

In [32]:
S = SpatialData(
    points={"transcripts": points},
    shapes={"cell_boundaries": shapes_model},
    tables={"table": table},
)

# Avoid print(S) for whole-slide SpatialData with SpatialData 0.7.2 + Dask 2026.1.1:
# __repr__ can fail while inspecting the Dask backing graph.
def inspect_sdata(sdata):
    print("SpatialData loaded/constructed successfully")
    print("Points :", list(sdata.points.keys()))
    print("Shapes :", list(sdata.shapes.keys()))
    print("Tables :", list(sdata.tables.keys()))
    print("Images :", list(sdata.images.keys()))
    print("Labels :", list(sdata.labels.keys()))
    print("Coordinate systems:", sdata.coordinate_systems)
    for n, t in sdata.tables.items(): print(f"Table {n}: {t.n_obs:,} x {t.n_vars:,}")
    for n, sh in sdata.shapes.items(): print(f"Shapes {n}: {len(sh):,}")
    for n, pt in sdata.points.items(): print(f"Points {n}: columns={list(pt.columns)}")

inspect_sdata(S)

WRITE_SDATA = False
SDATA_PATH = PROJECT_ROOT / "data/spatial/GSM9046088_CosMx.zarr"
if WRITE_SDATA:
    SDATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    S.write(SDATA_PATH, overwrite=True)
    print("Saved:", SDATA_PATH)

SpatialData loaded/constructed successfully
Points : ['transcripts']
Shapes : ['cell_boundaries']
Tables : ['table']
Images : []
Labels : []
Coordinate systems: ['global']
Table table: 43,565 x 1,010
Shapes cell_boundaries: 43,565
Points transcripts: columns=['x', 'y', 'target', 'y_local_px', 'fov', 'cell_ID', 'x_local_px', 'CellComp', 'Unnamed: 0']
